# Context guide

This notebook explains what `Context` represents in `spacecore`, how it is used by spaces and operators, what the `enable_checks` flag controls, and how context is chosen when you create new objects.

A useful mental model is

$$
\texttt{BackendOps} \to \texttt{Context} \to \texttt{Space} \to \texttt{LinOp}.
$$

- `BackendOps` describes **how numerical operations are performed**.
- `Context` describes **which backend, dtype, and checking policy are active**.
- `Space` and `LinOp` carry that context and use it to manipulate arrays and validate input data.


## 1. What `Context` signifies

`Context` is the runtime numerical policy attached to library objects.

More precisely, a context is the triple

$$
\texttt{Context} = (\texttt{ops},\ \texttt{dtype},\ \texttt{enable\_checks}).
$$

So a context answers three questions:

1. **Which backend is active?** For example `NumPy` or `JAX`.
2. **Which dtype should be used?** For example `float32`, `float64`, or `complex128`.
3. **Should runtime validation checks be enforced?**

Thus `Context` is not itself a space or an operator. It is the object that carries the numerical rules under which those objects live.


In [1]:
import numpy as np

from spacecore.backend import Context, NumpyOps, JaxOps

ctx_np = Context(NumpyOps(), dtype=np.float64, enable_checks=True)
ctx_jax = Context(JaxOps(), dtype=None, enable_checks=False)

print(ctx_np)
print(ctx_jax)

Context(ops=NumpyOps(family='numpy'), dtype=dtype('float64'), enable_checks=True)
Context(ops=JaxOps(family='jax'), dtype=<class 'jax.numpy.float32'>, enable_checks=False)


/Users/pavlopelikh/Documents/Dev/SpaceCore/spacecore/backend/jax/_ops.py:90: UserWarning: jax_enable_x64 is set to False, so default JAX dtype is set to float32. If you need float64, run `jax.config.update('jax_enable_x64', True)`.
  warn(


## 2. What belongs to a `Context`

A `Context` stores:

- `ops`: a concrete `BackendOps` implementation,
- `dtype`: the target dtype policy,
- `enable_checks`: whether runtime checks are active.

The backend controls how arrays are created and manipulated. The dtype controls what dtype conversion methods such as `asarray` and `assparse` target. The checks flag controls whether validation guards are actually enforced.


## 3. Basic methods of `Context`

Typical methods are:

- `ctx.asarray(x)`
- `ctx.assparse(x)`
- `ctx.convert(x)`
- `ctx.assert_dense(x)`
- `ctx.assert_sparse(x)`

The first three perform conversion into the backend and dtype specified by the context.
The last two are explicit assertion helpers. They always validate the representation because calling an assertion is an intentional request to check the invariant.


In [2]:
ctx = Context(NumpyOps(), dtype=np.float64, enable_checks=True)

x = ctx.asarray([1, 2, 3])
print(x)
print(x.dtype)

ctx.assert_dense(x)

[1. 2. 3.]
float64


array([1., 2., 3.])

## 4. What the `enable_checks` flag controls

This flag controls whether runtime validation is enforced.

At the `Context` level, it directly affects methods such as

- `assert_dense(x)`
- `assert_sparse(x)`

If `enable_checks=True`, these methods verify that the given object has the expected representation for the active backend. If `enable_checks=False`, they simply return without enforcing that validation.

At the `Space` level, `check_member(x)` is also guarded by the same flag. In other words, a space only runs `_check_member(x)` when

$$
\texttt{space.ctx.enable\_checks} = \texttt{True}.
$$

So the checks flag affects both low-level representation checks and higher-level space membership checks.


### 4.1 Example: dense assertion

With checks enabled, trying to validate a non-dense object as dense should fail.

With checks disabled, the same call does not enforce that validation.


In [3]:
ctx_checked = Context(NumpyOps(), enable_checks=True)
ctx_unchecked = Context(NumpyOps(), enable_checks=False)

# A plain integer is not a backend-native dense array.
bad_x = 7

for label, ctx_current in [
    ("checks enabled", ctx_checked),
    ("checks disabled", ctx_unchecked),
]:
    try:
        ctx_current.assert_dense(bad_x)
    except Exception as e:
        print(f"With {label}:", type(e).__name__, e)

With checks enabled: TypeError Expected dense array for numpy, got int
With checks disabled: TypeError Expected dense array for numpy, got int


The point is not that integers should be passed around as arrays. The point is that explicit assertion helpers remain strict regardless of `enable_checks`.

So a practical interpretation is:

- `enable_checks=True`: automatic object-level membership checks are active in spaces, operators, and functionals.
- `enable_checks=False`: those automatic checks are skipped where constructors and methods honor the context flag.
- `assert_dense` and `assert_sparse`: explicit backend-representation assertions that always validate.


### 4.2 Example: space membership

Suppose `X` is a space with a shape constraint. Then `X.check_member(x)` only runs the actual validation logic when checks are enabled.

Schematically,

$$
\texttt{if X.ctx.enable\_checks: X.\_check\_member(x)}.
$$

So the checks flag also controls whether shape, structure, or representation constraints are enforced at the space level.


In [4]:
from spacecore.space import DenseCoordinateSpace

X_checked = DenseCoordinateSpace((2, 2), Context(NumpyOps(), enable_checks=True))
X_unchecked = DenseCoordinateSpace((2, 2), Context(NumpyOps(), enable_checks=False))

good = np.zeros((2, 2))
bad = np.zeros((3,))

X_checked.check_member(good)
print("Good member passed with checks enabled.")

try:
    X_checked.check_member(bad)
except Exception as e:
    print("Bad member with checks enabled:", type(e).__name__, e)

# With checks disabled, validation is skipped.
X_unchecked.check_member(bad)
print("Bad member did not trigger validation with checks disabled.")

Good member passed with checks enabled.
Bad member with checks enabled: SpaceValidationError Expected shape (2, 2), got (3,)
Bad member did not trigger validation with checks disabled.


**Important!** `enable_checks=False` by default because certain checks are computationally heavy (for instance, validation of matrix symmetricity).
It is assumed that user controls themselves the input.
The principal role of this feature is to help user catch the error should it be related to wrong inputs/dtypes.

## 5. Explicit context versus global default context

You can provide context in two ways:

1. **Explicitly**, by passing `ctx=...` when constructing objects.
2. **Implicitly**, by setting a global default context, which is then used when no explicit context is provided and no better context can be inferred.

The library exposes helper functions such as:

- `set_context(...)`
- `get_context()`

for controlling that global default.


In [5]:
from spacecore import set_context, get_context

print("Before:", get_context())

set_context(Context(NumpyOps(), dtype=np.float64, enable_checks=False))
print("After:", get_context())

Before: Context(ops=NumpyOps(family='numpy'), dtype=dtype('float64'), enable_checks=False)
After: Context(ops=NumpyOps(family='numpy'), dtype=dtype('float64'), enable_checks=False)


## 6. Context resolution order

When the library needs to choose a context, the intended priority is:

$$
\text{explicit context} \;\succ\; \text{inferred context from other objects} \;\succ\; \text{global default context}.
$$

More precisely:

1. If an explicit context is passed, it is used.
2. Otherwise, the library tries to infer a context from the other given objects.
3. If that inference fails, the global default context is used.

So explicit context always wins.


### 6.1 Inference from existing objects

Context inference can come from objects that already carry a `.ctx`, or from backend-native arrays recognized by registered backends.

If multiple objects are used for inference, they must be backend-compatible. If they have the same inferred dtype, that dtype is kept. If their inferred dtypes differ, the resulting inferred context uses

$$
\texttt{dtype} = \texttt{None}.
$$

Also, the inferred `enable_checks` flag is the logical conjunction of the inferred flags:

$$
\texttt{enable\_checks} = \bigwedge_i \texttt{ctx}_i.\texttt{enable\_checks}.
$$

So checks remain enabled only if all inferred contexts have checks enabled.


### 6.2 Practical rule of thumb

When constructing objects, the easiest rule to remember is:

- pass `ctx=...` when you want full control,
- rely on inference when you are composing objects that already carry a context,
- rely on the global default only for convenience.

So for important user-facing code, explicit context is usually the clearest option.


In [6]:
from spacecore.space import DenseVectorSpace

# Global default is used when no explicit context is provided and nothing else determines one.
set_context(Context(NumpyOps(), dtype=np.float64, enable_checks=False))
X_default = DenseVectorSpace((3,))
print("Context from global default:", X_default.ctx)

# Explicit context overrides the default.
X_explicit = DenseVectorSpace(
    (3,), ctx=Context(NumpyOps(), dtype=np.float32, enable_checks=True)
)
print("Explicit context:", X_explicit.ctx)

Context from global default: Context(ops=NumpyOps(family='numpy'), dtype=dtype('float64'), enable_checks=False)
Explicit context: Context(ops=NumpyOps(family='numpy'), dtype=dtype('float32'), enable_checks=True)


To read more on conversion policy, dtype switch policy, see the tutorial on conversion.

## 7. Why `Context` exists separately from `BackendOps`

`BackendOps` describes the backend interface itself.

`Context` packages that backend together with runtime policy. That is why objects usually carry a `Context`, not a bare `BackendOps` instance.

In other words, `BackendOps` says what operations exist, while `Context` says under what numerical policy those operations should be used.


## 8. Typical usage pattern

A standard workflow is:

1. choose a backend,
2. build a context,
3. create spaces and operators from that context,
4. optionally set a global default for convenience.

Schematically,

$$
\texttt{BackendOps} \to \texttt{Context} \to \texttt{Space/LinOp}.
$$


In [7]:
from spacecore.space import HermitianSpace

ctx = Context(NumpyOps(), dtype=np.complex128, enable_checks=True)
H = HermitianSpace(3, ctx=ctx)

print(H)
print(H.ctx)

Context(ops=NumpyOps(family='numpy'), dtype=dtype('complex128'), enable_checks=True)


## 9. Summary

`Context` is the numerical policy object of the library.

It determines:

- the backend,
- the dtype,
- whether runtime checks are enforced.

You can provide it explicitly, or let the library fall back to inference and then to the global default.

The resolution order is

$$
\text{explicit} \succ \text{inferred} \succ \text{default}.
$$

Finally, `enable_checks` controls whether representation and membership invariants are actively validated at runtime.
